# Steam's Gaming Recommender Algorithm
#### Recommender Systems 2026
*Reyna Geluk, Stan Bakker, Mieke Fraters & Aryan Müller*

*15708845, 15840530, 15778770 & 15631591*


![STEAMLOGO](steam_store.jpg)
> Figure 1: Steam Logo. [Source.](https://store.steampowered.com/)

## Introduction

Steam is an online game distribution platform owned by Valve Corporation, where companies, game developers and publishers
can sell and host video games for users to buy (1,2). As of 2025, the platform is home to around 117 thousand games with over 10 thousand new
games added each year, making it one of the largest digital distribution platform for PC games (3, 4).

Although Steam encourages smaller indie developers to create and publish games on the platform, and a small percentage of them are able to thrive for a while, the current Steam algorithm mainly boosts games 
that are already being played by users (6). This creates reinforcing loops: the algorihtm boosts the games that already have visibility even more, and at the same time silences 
games that lack visibility, regardless of if those hidden games would be enjoyed by users (5,6). Popular games that get more reviews, keep getting recommended to users (6). This causes a popularity bias (8).

Indie games often lack the traction, community, hype and  marketing budget that AAA games have, causing the algorithm to lessen the possibility of indie games being discovered (6). 

By giving indie games a little boost in the algorithm, they may also stand a chance against the AAA games in the algorithm. Due to larger companies usually having a lot of surrounding hype and communities around their games,
their chances of publishing a succesful game is way higher compared to indie games. By implementing an algorithm that boosts indie games, without punishing AAA games, indie games will have a bigger chance of succeeding, without diminishing the success of AAA games.

In this project, we've tried to create an algorithm similar to Steam's game recommender, while trying to make sure that indie games are not marginalized in the process.



## The current Steam algorithm and Valve Corporation's transparency

Valve Corporation is quite transparent as to what factors their algorithm looks at to determine which games are shown to users. Their algorithm relies on player interest, and users get recommended a combination of personalized content; based on their own behavior, and featured games; popular games that currently have a lot of hype surrounding them. The algorithm also looks at what games friends of a user are playing, or what region the user is in (7). So individual user behavior and crowd taste similarity play a role in Steam's algorithm.

Tags also play a big role in determining which games should be recommended. Tags are categories that games can fall into, such as `Singleplayer`, `Multiplayer`, `Combat` or `RPG`. This means that the content of the game also determines what users will see the game recommended (7).

Thus, for our algorithm, we've decided to create an algorithm that combines user behavior and item content to generate relevant recommendations, keeping both popular and indie game diversity in mind.

## Datasets

The only data used for this algorithm can be found available publically online, with a free license. 


### Steam reviews dataset
The dataset can be found [here.](https://www.kaggle.com/datasets/kieranpoc/steam-reviews/data) Last updated: 2024.


The dataset by Kieranpo’c on Kaggle contains around 114 million rows, with each row consisting of users and their like/dislike labels for specific games. Whenever a user writes a review about a game they own, Steam asks them to say whether they liked and recommend a game. This data is stored in the column `voted_up` and contains `1` and `0`. The other columns necessary for the algorithm are `recommendationid`, `appid`, `author_steamid`, and `author_num_reviews`.

Although having this much data at our disposal is useful in theory, our hardware sadly does not have the power to run the entire dataset, especially when considering the two-week timeframe for this project. Therefore, the data has been filtered to only contain relevant entries and columns from users with a minimum of seven reviews. Although setting that limit at four would be most optimal to filter out the lowtail users, GitHub can’t handle those large files. Filtering like this lessens the amount of bias compared to other filtering methods without destroying system memory, while keeping this project feasible. How this was done, can be found in [this file.](https://github.com/Reynaeg/RecSysFinalProject/blob/main/ReviewDatasetResizeClean.ipynb)

Another plus from using this dataset, is that Steam only allows users to review games they've actively owned. Although users may have different playtimes, with some outliers maybe only having a few minutes or thousands of hours of playtime, most reviews will be legitimate.

### Steam Games dataset
The dataset can be found [here.](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset) Last updated: January 2026.


For the content based filtering algorithm, data about video games and their genres, categories and tags were needed. Thus, the Steam Games Datasets from Martin Bustos was used. The dataset contains genre-related information about more than 122 thousand Steam games, grouped by games.

The dataset contains the columns `Categories`, `Genres`, `Tags`, `Estimated owners` and `Publishers`.  These columns were needed for the content-based filtering and indie/AAA games categorization. 

There was an error in the dataset, offsetting values by one column. This problem was fixed before the algorithm was created, in Excel, by renaming the needed columns to their true respective names.

In [1]:
#Imports
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
df = pd.read_csv(r"C:\Program Files\allFilteredAggr7.csv") # Put the dataset in this specific path if you want to use it. Same goes for the other csv imports

#UnicodeDecodeError: 'utf-8' codec can't decode byte 0x99 in position 22574: invalid start byte
#Bla Bla throw more code at it till we don't get that error
games_df =  pd.read_csv(r"C:\Program Files\games.csv", sep=";", encoding="latin-1") #change the name depending on what it is called. 
games_df = games_df.rename(columns={"AppID": "appid"})
#also please call the appid column tags_appid... please.... and tags tags_tag (　＾▽＾)

#The part for the ethical part of the thing
games_filtered =  pd.read_csv(r"C:\Program Files\games_filtered.csv", sep=";", encoding="latin-1") #change the name depending on what it is called. 
games_filtered = games_filtered.rename(columns={"AppID": "appid"})
games_filtered = games_filtered.drop(columns=['id'])

## Creating the recommender algoirhtm

### Pushing the indie games
For this project, it was decided to build a recommender algorithm based on a content-based filtering algorithm and a KNN algorithm. Using these algorithms, recommendations for other games will be made based on reviews from users. However, to counter popularity bias, it is necessary to give smaller games a slight boost so they are recommended more often. The recommender system itself will therefore be a combination of content-based filtering, based on game categories, and a KNN model, where games reviewed by users with similar behavior will be recommended. Initially, each game is given an equal chance of being recommended to users. However, larger games still have a higher chance of being recommended, as they generally have more downloads, larger publishers, and therefore more reviews. In the following code block, functions will be written to determine the number of reviews per game, the average number of downloads per publisher per game, and the number of downloads per game. All three of these factors will be used to support and promote smaller, indie games.

In [3]:
#The df's of average owners per publisher, amount of owners per game and amount of reviews per game
#Aka how many players on avrage does the publisher have? It is numbers in the form of a string
def get_average_owners(owner_string):
    #replace the comma with a space
    owner_string = str(owner_string).replace(',', '')
    # split on the - in the string ( it is a range between low - high)
    low, high = owner_string.split('-')
    #Then return the int from low+high/2
    return (int(low) + int(high)) / 2
    
#we copy the dataframe to preserve the normal dataframe
df2_selected = games_filtered.copy()
#we grab the estimated owners per game and change it to the avg owners per game
df2_selected.loc[:, 'avg_owners_per_game'] = df2_selected.loc[:, 'Estimated owners'].apply(get_average_owners)
#We change the publishers to be strings split on the ,
df2_selected.loc[:, 'Publishers'] = df2_selected.loc[:, 'Publishers'].str.split(',')
#Replace the lists that were made with the split to their own rows for each publisher
df_exploded = df2_selected.explode('Publishers')
#Then remove the white spaces using strip
df_exploded.loc[:, 'Publishers'] = df_exploded.loc[:, 'Publishers'].str.strip()
#Then we Group by the publishers and then grab the mean for the avg owners per game
developer_scores = df_exploded.groupby('Publishers')['avg_owners_per_game'].mean().reset_index()
#Then we sort on avg owners per game descending.
developer_scores = developer_scores.sort_values(by='avg_owners_per_game', ascending=False)

#This is to find the avr player per game
#Create a new dataframe with only the appid
df_cleaned = df2_selected[['appid']].copy()
#then create ownersint and fill it with the estimated owners that went through the get_average_owners
df_cleaned['owners_int'] = df2_selected['Estimated owners'].apply(get_average_owners)
#sort the dataframe
df_cleaned = df_cleaned.sort_values(by='owners_int', ascending=False)

#count how many appid there are so we know how many reviews there are per game
df_counts = df['appid'].value_counts().reset_index()
df_counts.columns = ['appid', 'review_count']



#Change candidate scores using avg owners per publisher per game, amt owners per game and amt reviews per game


def adjust_scores_for_fairness(candidate_scores, alpha):
    #grab a copy of the candidate score
    fair_scores = candidate_scores.copy()

    #Create a dict from the dataset that has the avr owners per game
    owners_dict = df_cleaned.set_index('appid')['owners_int'].to_dict()
    #Create a dict from the dataset that has the review count per game
    review_dict = df_counts.set_index('appid')['review_count'].to_dict()
    #a dict with the avg owners per publisher
    dev_avg_dict = developer_scores.set_index('Publishers')['avg_owners_per_game'].to_dict()

    #grab the highest value from each dict
    max_owners = max(owners_dict.values())
    max_reviews = max(review_dict.values())
    max_dev_avg = max(dev_avg_dict.values())

    #loop through all the games in the far_scores df
    for appid in fair_scores:
        #Give the 3 parts a score. The highest a 1 and the lowest a 0. It is based on the value devided by the highest value
        owners_score = 1 - (owners_dict.get(appid, max_owners) / max_owners)
        review_score = 1 - (review_dict.get(appid, max_reviews) / max_reviews)   
        dev_score = 1-(dev_avg_dict.get(appid, max_dev_avg) / max_dev_avg)                 

        #Add all three the scores togheter and devide them by 3
        fairness_factor = (owners_score + review_score + dev_score) / 3

        #lend original candidate score with fairness factor, alpha controls how much the fairness adjusts the score
        fair_scores[appid] = (1 - alpha) * fair_scores[appid] + alpha * fairness_factor
    #Return all games sorted from highest to lowest adjusted score
    return dict(sorted(fair_scores.items(), key=lambda x: x[1], reverse=True))

### The code for pushing indie games
In the first part of the code above for boosting small games, three different dataframes are created. The dataset used for this code is the `games` dataset. From this dataset, information was used about the estimated number of owners, the publishers of each game, and the corresponding app ID. After the dataset was adjusted so that all columns were in the correct place, as discussed earlier under the section “Steam Games dataset,” the data still needed to be processed properly.

In the `games` dataset, there is a column called `Estimated owners`. The information in this column consists of a range representing the estimated number of owners a game might have. The values were strings in the format “0-1000”. Therefore, it was first necessary to convert this into an integer. In the function `get_average_owners`, the dash is removed from each value and the average of the range is calculated, meaning the highest number plus the lowest number, divided by two.

After that, the first dataframe is created. This is a dataframe for the average number of owners per publisher per game. For this, a new column called `avg_owners_per_game` is created, in which the average is calculated of all owners across all games that a publisher has. Because many games have multiple publishers, these are included separately in the dataframe, but with the same number of owners. It is important to look at the average number of owners per game per publisher, because this allows games from smaller publishers to be boosted later. The second dataframe is created for the estimated number of owners per game. Again, the function `get_average_owners` is used to replace the strings with integers. This makes it possible to boost games with fewer owners later. The third dataframe is created based on the large dataset of Steam reviews. In this dataset, the number of reviews per game is counted, allowing games with fewer reviews to be slightly boosted.

The final part of the code is the most important. The function `adjust_scores_for_fairness` takes a dictionary containing a recommendation score for each game and adjusts these scores based on the number of reviews, owners, and the popularity of the publishers. First, a dictionary is created for each of the three dataframes. Then, the highest value from each dictionary is taken in order to calculate a relative score. Next, these scores are calculated based on the highest value in each dataframe. The highest possible score is 1, which means that all aspects of the game are small, and the lowest score is 0, which means that a game has the most reviews, the most owners, and the most popular publishers. Then, the average of these three relative scores is taken.

The scores produced by the recommender, without taking popularity bias into account, are multiplied by this relative value so that smaller games are slightly boosted. How much these games are boosted can vary and depends on the size of the alpha parameter, which can always be adjusted. For now, this value is 0.5.

In the end, a new dataframe is returned in which the ranking of the games has been adjusted based on the number of reviews, the number of owners, and the popularity of the publishers.

### Explaining the algorithm
So, for the algorithm  we used a hybrid approach, combining collaborative filtering and content-based filtering to generate game recommendations.

The collaborative filtering uses a User-User K-Nearest Neighbours (KNN) model. A user-app matrix is constructed where each row represents a user, and each column represents a game. Each cell is then filled with values ranging from 1 (positive review), -1 (negative review), or remains empty (no review). Then cosine similarity is used to identify users with similar voting patterns. The assumption this is based on is that users who agreed on many games in the past are likely to agree on unseen games as well; it's assumed they have a similar taste.

The content-based filtering part uses TF-IDF with cosine similarity. Each game's tags, categories and genres are combined into a single string. TF-IDF then assigns higher weights to tags that are rare across the dataset and lower weights to tags that appear in nearly every game. This means games sharing uncommon, distinctive tags (like "3D Fighter") are ranked as more similar than games that only share generic ones (like "Single player"). Cosine similarity is then computed between all pairs of tag vectors.

The hybrid recommender combines both parts by first identifying similar users, collecting the games those users liked and then finding games with similar tags to that collection. Each candidate game receives a score based on how many times it appears across these tag comparisons. Games the target user has already reviewed, both positive and negative, are removed and the remaining candidates are ranked by score.

In [4]:
# ═════════════════════════════════════════════════════════════════════════════
# 0. MY NOTES AND THOUGHTS
# ═════════════════════════════════════════════════════════════════════════════
#So I need to make:

# Collaborative filtering: Users like you also liked. Using players with similiar voting patterns and stuff. (User-User KNN)
# Content- based filtering: Games similar to ones you liked. We are gonna use the tags. (TF-IDF + Cosine Similarity)
# Hybrid recommender: combine the two from above
# I will just be figuring out what the dude did and copy it so that is why the code might be a bit weird? like wtf are all those cat things
# They make my head spin with the logic behind it XD oeh I am gonna use the fancy code barriers
# Oh yea I used printing inbetween to see what it was doing and if going wrong where


# ═════════════════════════════════════════════════════════════════════════════
# 1. MAKING THE CODE USABLE OR SOMETHING
# ═════════════════════════════════════════════════════════════════════════════

# Map users and apps to contiguous integer indices.
#So bassically what this part does. It grabs the information from the columns. Tells panda to see it as type "category" aka unique labels not just numbers
#And apparently cat.codes replaces the unique label with a small integer assigned in sorted order. So no more huge numbers? idk
# So .loc[:. "name"] is need or else pandas starts complaining "A value is trying to be set on a copy of a slice from a DataFrame." it legit told me
# To do it this way
df.loc[:, "user_idx"] = df["author_steamid"].astype("category").cat.codes
df.loc[:, "app_idx"]  = df["appid"].astype("category").cat.codes

# .cat.categories is the reverse lookup: a sorted array of the original IDs, where position i holds the original ID that was assigned cat.code i.
# We save this so we can convert back from matrix indices to real IDs later.
unique_user_ids = df["author_steamid"].astype("category").cat.categories  # index is steamid
unique_app_ids  = df["appid"].astype("category").cat.categories           # index is appid

n_users = df["user_idx"].max() + 1   # total rows in our matrix
n_apps  = df["app_idx"].max() + 1    # total columns in our matrix

#this is just a check
print(f"Unique users : {n_users:,}")
print(f"Unique apps  : {n_apps:,}")


# ═════════════════════════════════════════════════════════════════════════════
# 2. COLLABORATIVE FILTERING  (User-User KNN)
# ═════════════════════════════════════════════════════════════════════════════
# so the matrix is going to look like this:
# Each row is a user
# Each column is a game
# And every cell holds a vallue of 1 = liked, -1 = disliked or missing = no review

# A sparse matrix is used because otherwise we are going to get memory issues again? anyway since a lot of users have only reviewed a view of the games
# this type of matrix only stores not zero values.
    
# Convert voted_up (1/0) to explicit +1 / -1 ratings.
# This ensures thumbs-down reviews are stored as -1 (non-zero) in the sparse
# matrix, rather than 0 which would make them invisible and indistinguishable, from "never reviewed".
# Also same reason for the .loc[:, "]
df.loc[:, "rating"] = df["voted_up"].map({1: 1, 0: -1})


#once again just to show it is working on making the matrix aka which step it is at
print("Building user-app sparse matrix ")

# csr_matrix takes three things:
# (values, (row_indices, col_indices))  and  shape
# Each review row contributes one non-zero cell so the 1 (liked) and -1 (disliked). Cells that are absent entirely (true zeros) mean "never reviewed".
# Also I used csr_matrix instead of the coo_matrix the dude used because. Appently coo is faster to build but slower to do things row by row?
# And that is basically how the KNN stuff works so this works faster for that.
user_app_matrix = csr_matrix(
    (df["rating"].values,            # so the 1 or -1.
     (df["user_idx"].values,         # which row (user)
      df["app_idx"].values)),         # which column (app)
    shape=(n_users, n_apps)
)

#Then the fit the model part 
# so metric = cosine means similarity is based on angle between vectors, not magnitude
# and algoritm = brute means compares against every user
model_knn = NearestNeighbors(metric="cosine", algorithm="brute")
# and then fit the matrix with it. Apparenlty this makes it answer quicker for like query time?
model_knn.fit(user_app_matrix)

# Time to announce which step we are at ヽ(￣▽￣)ノ
print("KNN model fitted.")



# Time to actually write the KNN code.... YIPPIE
# I grabbed the nearest 5 for the test. but for the end funtion we have to remove it!
# Also remember user_idx is that cat code version of the user id
def get_similar_users(user_idx: int, n_neighbors: int):
    #I made row it's own thing because in the dudes code it was in the kneighbors thing but I wanted it seprated to see better what is going on where.
    row = user_app_matrix.getrow(user_idx)   # extract this user's row as a sparse vector. (This basically also doesn't store the 0... save the pc's)
    distances, indices = model_knn.kneighbors(row, n_neighbors=n_neighbors)
    similar_idxs = indices.flatten()[1:]     # [1:] skips the user themselves since like they are 0
    # Convert integer indices back to real steam-IDs using our reverse-lookup array. (aka this gives the true ID not the cat one)
    return [unique_user_ids[i] for i in similar_idxs]
    # ^you need the [] around it otherwise it gives you an error.

# a quick test just for the fun. And to see if it works (oh whoops my bed another test works and all that stoof now go poof)
#sample_user_idx = 0
#ample_steamid  = unique_user_ids[sample_user_idx]   # real ID for index 0
#similar_users   = get_similar_users(sample_user_idx)
#print(f"Similar users to steam-ID {sample_steamid}:")
#print(similar_users)
# Yippie it does print things


# ═════════════════════════════════════════════════════════════════════════════
# 3. CONTENT-BASED FILTERING  (TF-IDF + Cosine Similarity)
# ═════════════════════════════════════════════════════════════════════════════
# Okay so the idea behind this one. Smack the two dataframes togheter on the appids.
# Then make all the tags just one long string
# Let TF-IDF lose on those strings. commons tags like "singleplayer" or something really normal get low weight
# But rarer tags like "shooter" (idk what is rare okay so bare with me CS:GO was the first game in the list) get a high weight.
# This makes games with similiar high weight tags closer to eachother then if they would be sharing a low weight one.
# And then grab the cosine similarity from all the tag vectors pairs. The more game overlap in tags the closer to 1 they get.
# Oh btw this uhm... is more my frankenstein of a code then the dudes code since his worked on discription and ours goes for the tags.

# Update time~
print("Transforming all tags to strings")

# OKAY so since our csv had 3 columns worth of tags (likes) we diced to use. we gotta chance plans
# First no clue if there is NaN make it an empty string. No errors or crashes here.
# Then split each column on commas since that is how they are build. Get the tags alone.
# Strip the whitespaces they are probably there. also make everything lowercase!
# Comine all 3 list of tokens into one. Use set so there are not dublicates
# Then we create one long string seperated by spaces so my old code still works!

def split_col(item):
        # Split on the comma in the string so you make into a list of stripped tokens.
        # Returns an empty list if the value is missing. just so no crashes
        if pd.isna(item) or item == "":
            return []
        #turns item into a string then loops through it on t split on the ,
        return [t.strip() for t in str(item).lower().split(",")]

#explenation up there ^
def build_tag_string(row):
    tokens = (
        split_col(row["Categories"]) +
        split_col(row["Tags"]) +
        split_col(row["genre"])
    )
    # set() removes duplicates; sorted() keeps the order
    unique_tokens = sorted(set(tokens))
    #create an string and add to it so it returns the long string
    return " ".join(unique_tokens)

#create another row in the games df with the tag_string. apply like goes over the columns and uses the def created above. axis 1 is columns
games_df["tag_string"] = games_df.apply(build_tag_string, axis=1)

# Now we couple the strings to the app id's in a dataframe ^-^
# Also I made it games with no tags (I think those exist) that just gives an empty string so TF-IDF still produces a row for them... just zeros
app_tag_df = (
    pd.DataFrame({"appid": unique_app_ids}) #makes the dataframe with the unique app ids created in the first part
    .merge(games_df[["appid", "Name", "tag_string"]], on="appid", how="left") #merges the strings in on the left based on appid
    .fillna({"tag_string": "", "Name": "Unknown"})  # games with no tags get an empty document and name is unknown for the end result thingy
)

# Keep a name lookup Series or something. appid linked to game name, for readable output later when we return stuff
appid_to_name = app_tag_df.set_index("appid")["Name"]

# Separate Series of just the tag strings, for TF-IDF (idk how to rewrite the code for the update time and I am too tired for that right now
# So deal with this
app_tag_strings = app_tag_df["tag_string"]

#it is update time (〜^∇^)〜 (it works so notes now
#print(f"Games with tags    : {(app_tag_strings != '').sum():,}") #shows how many games had tags cuz the string isn't empty
#print(f"Games without tags : {(app_tag_strings == '').sum():,}") #Shows how many gmaes had no tags... thus similarity will be 0 cuz there is nothing

#print("Building TF-IDF matrix from game tags")

# So Tfidfectorizer is a scikit-learn thing that coverts reaw thext into a numerical matrix of TF-IDF features.
# So analyzer = word treat each tag as a word token. Not the characters. 
# ngram_range=(1, 2) consider single tags AND pairs of adjacent tags. This lets the model learn that "Open World" together 
# is more meaningful than "Open" and "World" separately.
# Yea I don't get that one a lot either but think it just makes sure that if there was a space in the tag it grabs those words togheter instead of
# Making them two seperate ones?
# min_df=2 is so that it ignores if there is only one tag in the entire dataset. Since well nothing to compare it to so useless information
# also there is a thing you can add called (stop_words="english") but like if there is a stopword in the tags I think it would still be usefull?
# it is not as if we are trying to compare books or something
tf = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2)
tfidf_matrix = tf.fit_transform(app_tag_strings)
# The result is a sparse matrix of shape (n_apps, n_unique_tag_features)

#update time (｢• ω •)｢ wow look how far we already got... okay me this is all me I can't tell if I am losing my mind
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


# it is callable function time again~ time to find games with similar tags. Oh wow this part I can steal off the dude and I have no idea what it does
# so apparently linear_kernel is equivalent to cosine similarity. But only when the vectors are already "L2-normalised" which is the case in tf-idf vectors
# Anyway the function grabs the cat app ids. And then same as by the other one 5 other neighbours
# And this functions returns a list of unique appids that are the most similiar to the one we had
def get_similar_apps(app_idx: int, n_neighbors: int):
    # Compute similarity between this one app and ALL other game in one go. Cuz apparently we can do that now?
    # Result is a 1D array of similarity scores, one per game
    sims = linear_kernel(tfidf_matrix[app_idx], tfidf_matrix).flatten()

    # So argsort does what it does. It returns indices that would sort the array in ascending order.
    # That is also why we need [::-1] cuz reverses it to descending (highest similarity first).
    # [1:n_neighbors] skips index 0 (the game itself) cuz no we don't need to recommend the game itself you know.
    indices = sims.argsort()[::-1][1:n_neighbors]

    # Convert integer indices back to real appids. aka not the cat version
    return [unique_app_ids[i] for i in indices]

# Also another quick test time =＾● ⋏ ●＾= (look it is a cat.. it is staring into my soul) (it worked so notes now)
#sample_app_idx = 0
#sample_appid   = unique_app_ids[sample_app_idx]
#similar_apps   = get_similar_apps(sample_app_idx)
#print(f"Apps similar to appid {sample_appid}: {similar_apps}")

# ═════════════════════════════════════════════════════════════════════════════
# 4. HYBRID RECOMMENDER (time to mush those two monsters into one)
# ═════════════════════════════════════════════════════════════════════════════
# So this part grabs both of the 2 and 3. And makes them work togheter.
# This is because each part has it own weakness:
# Collaborative filtering doesn't really recommend games with a lot of reviews. so it won't appear a lot if at all.
# And content based only recommend games simliar to what the user already knows. Not really a lot of variety.
# Mixing both of these we get games recommended from users with a simliar taste. 
# Grab the games those users gave a thumbs up. 
# Then add like games that have simliar tags
# Look how many times a game apears. the more it apears the stronger the recommendation.
# Remove games we know the user has already reviewed. And return the top however many we want!
# That probably sounds easier to do than to make.

# update time! ʘ‿ʘ 
print("now creating the callable feature to recommend games with")
# Now creating the callable feature~ 
# So first input the steam id of the user we want to make a recommendation for (NOT THE CAT VERSION JUST THE LONG STRING)
# Then like the amount of simliar users we want and the simliar apps and how many recommendations we want. You know the good stuff.
# And like I show. I returnes a list of the recommended appids. soreted by hybrid score most recommended first.
def recommend_apps(target_steamid, n_similar_users: int, n_similar_apps: int, top_n: int) -> list:
    # First step. Check if wanted user is even in our system!
    # We have the unique_user_ids. So check if userid in there otherwise throw a hissy fit.
    # Then use .get_loc() to grab convert the steamid into the cat.code stuff so we can use it as a row index in the user-app matrix we made.
    if target_steamid not in unique_user_ids:
        raise ValueError(f"steam-ID {target_steamid} not found in filtered dataset. Please try again")
    # So it only gets to this step if it didn't raise the error. Just so you guys understand the error breaks out of this function. (pretty sure it just
    # ends the program but I would need to test that when Aryan gives me the game dataset with tags and everything works).
    user_idx = unique_user_ids.get_loc(target_steamid)

    # First step after first step! find similar users aka collaborative filtering
    # The plus one is cuz the first one would be the user itself.
    similar_steam_ids = get_similar_users(user_idx, n_neighbors=n_similar_users + 1)

    # Next step! collect games those users liked!
    # create a set to save the data in
    seed_apps = set()
    # then loop through the dataframe finding all the reviews that were positive.
    for sid in similar_steam_ids:
        liked = df[
            (df["author_steamid"] == sid) &
            (df["voted_up"] == 1)
        ]["appid"].unique() #this makes sure it grabs the appid but that it is unique
        seed_apps.update(liked) #then adds it to the set. 

    # Now the next step now we got the liked games. This is where the tags stuff comes in.
    # Candidate_scores tracks how many "votes" each candidate game has received.
    # Every time a game appears as content-similar to a seed, its score goes up by 1.
    # so create the dict that keeps tract of the game and what their score is
    candidate_scores: dict = {}

    # then start looping through the appid's we got from the users liked list (probably a lot of games since it is from all the users and stuff)
    for appid in seed_apps:
        # check if the appid is in our list (just to make sure since we did filter data out)
        if appid not in unique_app_ids:
            continue   # skip if this game wasn't in our filtered data
        # Then we grab the cat version of the appid if was
        a_idx = unique_app_ids.get_loc(appid)
        # Then we get like the games with a lot of simliar tags!
        for sim_appid in get_similar_apps(a_idx, n_neighbors=n_similar_apps + 1):
            #Then we grab the dict and either create a entry and add a point for the game. Or just add a point for the game.
            candidate_scores[sim_appid] = candidate_scores.get(sim_appid, 0) + 1

    # great now we have dict full of games with simliar tags to the games the nearest neighbour users had... Does this still make sense to you guys?
    # I think I am doing pretty well at explaining... and praying this actually works cuz I can't test it.

    # anyway next step is removing games the user knows (∿°○°)∿ 
    # We do this for every game the user knows. So be it a negative or a positive review!
    # So we create a set with all the know steam reviews appid's
    already_reviewed = set(df[df["author_steamid"] == target_steamid]["appid"].unique())
    # Then we loop the dict over this set. Something something oh didn't we have this by networkscience last year. Fun. 
    # anyway this makes it so the dict stuff only saves in the dict again if it isn't already in the set above.
    candidate_scores = {
        k: v for k, v in candidate_scores.items() if k not in already_reviewed
    }
    
    #Now to adjust the score to fainess using the code from before this block
    #This makes it so indie games get an extra boost compared to AAA games.
    fair_scores = adjust_scores_for_fairness(candidate_scores, alpha=0.5)
    
    # now let them battle it out (aka rank them) and return the top however the many
    #sorted does you know. sort. reverse makes sure it is highest on top. And well we sort the dict based on the key.
    ranked = sorted(candidate_scores, key=candidate_scores.get, reverse=True)
    ranked_fair = sorted(fair_scores, key=fair_scores.get, reverse=True)
    
    #and then return until the top_n we gave. in a human readable way plus the apid (as int cuz damn) oh whoops forgot the cut off fixed now
    #okay had to change this to two new lines of code so that we can return both:
    candidate_result = [(appid_to_name.get(a, "Unknown"), int(a)) for a in ranked[:top_n]]
    fair_result = [(appid_to_name.get(a, "Unknown"), int(a)) for a in ranked_fair[:top_n]]
    
    return candidate_result, fair_result

print("finished set up ready for testing")

Unique users : 2,957,680
Unique apps  : 101,212
Building user-app sparse matrix 
KNN model fitted.
Transforming all tags to strings
TF-IDF matrix shape: (101212, 14144)
now creating the callable feature to recommend games with
finished set up ready for testing


### Explanation of output
The algorithm gives two outputs. These are two ranked lists of recommended games. The first list uses the raw hybrid scores without any alterations. The second list has the fairness adjustments code applied to it. The fairness adjustment as explained above slightly adjust the order of the list boosting games with fewer reviews, reducing the tendency to over-recommend popular AAA titles at the expense of smaller indie games. Both lists return the top N recommendations as game name and app ID pairs.

Below is the code where the algorithm gets called and actually gives the lists of the output. The code uses random users from the data frame, but it is also possible to put in your own Steam user ID. If the user ID is not included in the dataset, you'll get a value error. With enough memory, this algorithm can run on the original, unfiltered dataset, including users with seven or less reviews.

## Results

In [5]:
#I am lazy just give me random users from the list to test it with
random_user = df["author_steamid"].sample(1).values[0]

#recommend_apps(target_steamid, n_similar_users: int, n_similar_apps: int, top_n: int) -> list: (so I know what to fill in)
candidate_recs, fair_recs = recommend_apps(random_user, 50, 50, 10)
print(f"For user: {random_user}, we recommend these games:")
print("Candidate Scores (unajusted)")
print(candidate_recs)

print("Fair Scores (adjusted)")
print(fair_recs)

#good news! valueerror works. Sad news, I am not in the dataset :( 

For user: 76561198254876761, we recommend these games:
Candidate Scores (unajusted)
[('Metro Exodus', 1449560), ('Asteroid', 2020850), ('Muscle MILF', 2315610), ('Unsung Warriors - Prologue', 959080), ("Manga Maker's Mega Milkers", 2291810), ('Kinky Cosplay: Gyarus Gone Wild', 2255420), ('Pirate Booty and the Bukkake Buccaneer', 1897820), ('Lewd Life with my Doggy Wife', 1991480), ('Aristocunts II Re:ERECTION', 2017700), ('Netorious Neighbor Cumming for their Wives!', 1851000)]
Fair Scores (adjusted)
[('Asteroid', 2020850), ('Metro Exodus', 1449560), ('Muscle MILF', 2315610), ('Unsung Warriors - Prologue', 959080), ('Kinky Cosplay: Gyarus Gone Wild', 2255420), ("Manga Maker's Mega Milkers", 2291810), ('Lewd Life with my Doggy Wife', 1991480), ('Pirate Booty and the Bukkake Buccaneer', 1897820), ('Aristocunts II Re:ERECTION', 2017700), ('Netorious Neighbor Cumming for their Wives!', 1851000)]


### Output results
As seen above if the code runs it gives two separate lists with pairs. The output `Candidate Scores (unadjusted)` gives output without boosting indie games, and the output `Fair Scores (adjusted)` boosts indie games.

The lists show the name of the game and the app ID, however if the game name was not in our dataset, it shows `Unknown` as the name. Often, these are expansions or DLC for already published games. It is still very possible to find the name of the `Unknown` value in the steam store using the app ID, that is why both name and ID are shown. By writing the ID value after the URL https://store.steampowered.com/app/ the game can be found on the Steam store.

## Evaluating 

Evaluating a recommender system purely on user reviews is difficult. As stated, the dataset used for the algorithm contains reviews from users, which it bases its recommendations on. In a perfect world, every user would leave a review for every game they've ever played, but this is unfortunately not the case. Thus, the use of standard evaluating metrics like f1-score will not accurately reflect the results of the algorithm. A game the recommender suggest might be sitting in the user's library already, or it might be something they would genuinely love, but there is no way to know from the data alone. 

Since there are also no ratings, and the recommender outputs a ranking, the decision was made to utilize a method known as **Hit Rate@K**. The core idea behind Hit Rate@K is that for each test user, one known positive interaction is hidden from the model so that it cannot use this item whilst it generates its recommendations (9). A "hit" is then recorded if this same hidden item appears anywhere in the recommendations generated by the model. In this project, one positive review of a game is hidden per user, then the user-app interaction matrix is rebuilt without those hidden reviews, and the recommender is asked to produce a ranked list of (in this evaluations case) 25 games, where if the hidden game appears, it counts as a hit. 

This method at least allows us to see if hypothetical users would see a game they find interesting in their recommendations or not. The recommender does however make a prediction based upon about 100,000 unique games, and only outputs the top 25, as any higher number would probably not be seen by the user at all. Ultimately, the recommender, which can be found in [this file about the evaluation](https://github.com/Reynaeg/RecSysFinalProject/blob/main/RecSysEval.ipynb), ultimately shows results that range from ~5% to ~10% Hit Rate depending on the parameters set, though usually in rare cases does it hit high. Meaning that in a few cases, users were most definitely recommended a game from their list of reviews, but in a lot of cases they were also recommended new games that were similar to the games they have reviewed, but not one we can guarantee they have. This can be due to there not being enough data, especially when hidden, as our review data does not contain all the data you would have in a real production setting, or that the game tags which identify similar games are too simple and give too large of a group to recommend. 


## Implentation of the algorithm and a small change to the Steam Store Layout:

The current Steam storefront could play a role in countering the popularity bias.

One of those ways could be by adding a section promoting smaller games and developers. Currently the Steam storefront worsens the popularity bias, by mainly promoting the already popular games or games with a lot of traction around them.

![STEAMLOGO](steam_store.jpg)
> Figure 2: PLACEHOLDER

By adding a section for smaller games right beneath the huge popular game adversitements, more people may discover indie games. This section should have a boosted alpha-value, promoting mainly indie games.
Steam's current algorithm could also boost indie games, by adding our changes but with a smaller alpha-value to make sure AAA games can still make a profit.

![STEAMLOGO](steam_store.jpg)
> Figure 3: PLACEHOLDER

## Conclusion and discussion

This project set out to build a game recommender for Steam that reduces popularity bias against indie games. The resulting hybrid system, combining user-based collaborative filtering with TF-IDF content-based filtering, seems to learn a bit of user preferences to generate recommendations that might be intriguing to users, and sometimes even a bit diverse. The fairness adjustment also showed smaller titles getting a higher chance of being shown alongside larger games without penalizing them.

The system's main limitations and low Hit Rate can be explained with having limited data, having only review data and no library, playtime, or other defining information, which can mean that the model is simpler than what a production system would have. With richer data and more computing power, the same architecture would likely perform considerably better, but still can recommend new and intriguing titles, if not giving accurate recommendations. 

#### Changes for a future project
If this project would be done again, here are some changes we suggest should be implemented.
- We've mainly used positive reviews to determine if a game should be recommended (`voted_up == 1`). However, the negative reviews could also be implemented to add negative weights to, for example, certain tags. Currently, the negative reviews are only counted to find games a user has already played, but they could play a bigger role in the workings of the algorithm.
- More time should be spent on adjusting parameters, to make recommendations as accurate as possible. 
- Using a dataset containing more direct information about users and their game statistics would help greatly in improving this model. Currently, the algorithm has no way of knowing what games a user may have in their library, so having better data could help.
- Due to the time constraint and hardware limitations, the dataset we used only contained users with more than seven reviews. Using the full dataset may increase the algorithm's accuracy. 

## Sources
1: Steam, the ultimate online game platform. (n.d.). https://store.steampowered.com/about/ <br>
2: Gamewijzer Redactie. (2025). Steam – wat moet je weten over dit gaming platform? Gamewijzer. https://www.gamewijzer.nl/steam/ <br>
3: https://icon-era.com/statistics/steam-game-statistics/ <br>
4: Ekanjo. (2025). Poll: Steam continues to reign on the PC gaming market. https://boilingsteam.com/poll-where-are-you-buying-the-majority-of-your-pc-games-right-now-in-2025/ <br>
5: Sulistyawan, I. (2025). The Hidden Gem Problem: Why Great Indie Games Die in Steam's Algorithm. *Medium*. https://medium.com/@ivan.adela/the-hidden-gem-problem-why-great-indie-games-die-in-steams-algorithm-3f0cbf4d23e7 <br>
6: Crichton-Stuart, E. (2025). Indie Games on Steam Make Up 48% of Revenue. *Games.gg*. https://games.gg/news/indie-games-on-steam-make-up-48-percent-of-revenue/ <br>
7: Peterson, E. (n.d.). Steam Visibility: How games get surfaced to players. *Valve*. https://steamcdn-a.akamaihd.net/steamcommunity/public/images/steamworks_docs/english/SteamVisibility.pdf
8: Dina - Meeting 2 - Data, Bias and Automation. (2026) https://github.com/uvapl/recommender-systems/raw/2025/materials/Dina%20-%20Meeting%202%20-%20Data,%20Bias%20and%20Automation.pdf 
9: Dadoun, A. (2023). How to Assess Recommender Systems: A deep dive on Evaluation Metrics Formulas. https://towardsdatascience.com/how-to-assess-recommender-systems-10afd6c1fae0/